In [ ]:
!pip install chardet -q

In [ ]:
import chardet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, csr_matrix
from sklearn.decomposition import TruncatedSVD

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, label_binarize, OneHotEncoder, StandardScaler
from sklearn.calibration import CalibratedClassifierCV

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    matthews_corrcoef, cohen_kappa_score, classification_report,
    confusion_matrix, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

RANDOM_STATE = 42
N_QUINTILES  = 5
print('Усі бібліотеки імпортовано успішно')

In [ ]:
with open('./data/appeals_2026-04-02.csv', 'rb') as f:
    enc = chardet.detect(f.read(100000))
print(enc)
        
df= pd.read_csv(
    './data/appeals_2026-04-02.csv',
    encoding=enc['encoding'],
    sep=';',
    parse_dates=['receivedDateTime']
)
print(f'Форма датасету: {df_raw.shape}')
df.head()

In [ ]:
df.dtypes

In [ ]:
print("Розмір датасету:", df.shape)
print(df.info())

In [ ]:
df["receivedDateTime"] = pd.to_datetime(df["receivedDateTime"], errors="coerce")

df["year"] = df["receivedDateTime"].dt.year
df["month"] = df["receivedDateTime"].dt.month
df["day"] = df["receivedDateTime"].dt.day
df["hour"] = df["receivedDateTime"].dt.hour
df["weekday"] = df["receivedDateTime"].dt.day_name()

In [ ]:
print(df["status"].value_counts())
print(df["status"].value_counts(normalize=True))

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_percent = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_df = pd.DataFrame({
    "missing": missing,
    "missing_%": missing_percent
})

missing_df

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(data=df, y="result", order=df["result"].value_counts().index)
plt.title("Розподіл результатів звернень")
plt.show()

рисунок 3.1 — Розподіл результатів звернень

графік 3.1 показує загальну статистику результатів опрацювання звернень громадян де чітко видно переважну більшість позитивно вирішених питань порівняно з кількістю наданих роз’яснень. Кількість позитивно вирішених справ становить понад десять тисяч тоді як кількість наданих роз’яснень знаходиться на рівні близько чотирьох тисяч звернень.


In [ ]:
df["result"] = df["result"].fillna("Невідомо")

In [ ]:
top_kinds = df["kind"].value_counts().head(15).index
df_k = df[df["kind"].isin(top_kinds)]

plt.figure(figsize=(10,6))
sns.countplot(data=df_k, y="kind", hue="result")
plt.title("Top kind vs result")
plt.show()

рисунок 3.2 — Розподіл результатів за видами проблем

графік 3.2 показує деталізацію результатів залежно від тематики звернення де найбільша кількість позитивних рішень спостерігається у сферах електропостачання та санітарного очищення міста. Категорії обслуговування ліфтів та водопостачання також демонструють високий рівень успішного вирішення питань. У той же час питання благоустрою та інші загальні звернення мають вищу частку наданих роз’яснень порівняно з іншими категоріями.


In [ ]:
top_orgs = df["organizationName"].value_counts().head(10).index
df_org = df[df["organizationName"].isin(top_orgs)]

plt.figure(figsize=(12,6))
sns.countplot(data=df_org, y="organizationName", hue="result")
plt.title("Топ організації vs результат")
plt.show()

рисунок 3.3 — Статистика результатів у розрізі організацій

графік 3.3 показує ефективність роботи різних комунальних підприємств та установ де найбільший обсяг позитивно вирішених заявок припадає на Вінницяобленерго. Комунальні підприємства Вінницяоблводоканал та Вінницяміськліфт також входять до лідерів за кількістю успішно закритих звернень. Деякі організації такі як Житлово-експлуатаційні об’єднання мають значну кількість наданих роз’яснень що може свідчити про специфіку їхньої діяльності.


In [ ]:
df["receivedDateTime"] = pd.to_datetime(df["receivedDateTime"])

df["date"] = df["receivedDateTime"].dt.date
df["hour"] = df["receivedDateTime"].dt.hour
df["month"] = df["receivedDateTime"].dt.month
df["dayofweek"] = df["receivedDateTime"].dt.dayofweek

In [ ]:
df.groupby("date").size().plot(figsize=(14,5))
plt.title("Кількість звернень по днях")
plt.show()

рисунок 3.4 — Динаміка кількості звернень по днях

графік 3.4 показує зміну активності громадян протягом першого кварталу 2026 року де зафіксовано значний сплеск кількості звернень у середині січня. Після пікового періоду коли щоденна кількість заявок сягала понад чотириста одиниць спостерігається поступова тенденція до зниження навантаження. Протягом лютого та березня графік демонструє стабілізацію активності з регулярними коливаннями які ймовірно пов’язані з днями тижня.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hour_counts = (
    df.groupby(["hour", "result"])
      .size()
      .reset_index(name="count")
)

hour_pivot = (
    hour_counts.pivot(index="hour", columns="result", values="count")
    .fillna(0)
    .sort_index()
)

hour_pivot.plot(
    kind="area",
    stacked=True,
    alpha=0.8,
    figsize=(18,6)
)

plt.title("Result по годинах")
plt.xlabel("Hour")
plt.ylabel("Count")
plt.legend(title="Result")
plt.show()

рисунок 3.5 — Розподіл результатів за годинами доби

графік 3.5 показує інтенсивність звернень протягом доби де основний пік активності припадає на десяту годину ранку. Протягом робочого дня з дев’ятої до шістнадцятої години спостерігається стабільно високий рівень звернень з невеликим спадом під час обідньої перерви. У вечірні та нічні години активність значно знижується досягаючи мінімальних значень о третій годині ночі.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df_feat = df.copy()

# ----------------------------
# 1. Feature engineering
# ----------------------------
df_feat["hour"] = df_feat["receivedDateTime"].dt.hour
df_feat["dayofweek"] = df_feat["receivedDateTime"].dt.dayofweek
df_feat["month"] = df_feat["receivedDateTime"].dt.month

df_feat = df_feat.dropna(subset=["result"])
df_feat["result_encoded"] = df_feat["result"].astype("category").cat.codes

# ----------------------------
# 2. Features
# ----------------------------
features = ["hour", "dayofweek", "month", "type", "kind", "status"]

df_model = df_feat[features + ["result_encoded"]]
df_model = pd.get_dummies(df_model, columns=["type", "kind", "status"], drop_first=True)

# ----------------------------
# 3. Correlation
# ----------------------------
corr = df_model.corr(numeric_only=True)["result_encoded"].drop("result_encoded")

# split
pos_corr = corr[corr > 0].sort_values(ascending=True)
neg_corr = corr[corr < 0].sort_values(ascending=True)

# ----------------------------
# 4. Plot
# ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# negative correlations
sns.barplot(
    x=neg_corr.values,
    y=neg_corr.index,
    ax=axes[0],
    palette="Reds_r"
)
axes[0].set_title("Негативна кореляція з result")
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set_xlabel("Correlation")

# positive correlations
sns.barplot(
    x=pos_corr.values,
    y=pos_corr.index,
    ax=axes[1],
    palette="Blues"
)
axes[1].set_title("Позитивна кореляція з result")
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_xlabel("Correlation")

plt.tight_layout()
plt.show()

рисунок 3.6 — Кореляція ознак з результатом

графік 3.6 показує фактори які мають найбільш вагомий статистичний зв’язок із позитивним результатом вирішення питання. Найсильнішу позитивну кореляцію демонструють звернення щодо благоустрою та конструктивних елементів будинку а також фактор місяця подання заявки. Найбільший негативний вплив на вірогідність отримання певного результату мають статус завершення справи та категорії обслуговування ліфтів чи електропостачання що може вказувати на складність вирішення таких питань.
